# SVM (RBF) Baseline Model
SVM (Support Vector Matchine)

**Module:** IT3051 - Fundamentals of Data Mining  
**Project:** Next-Year PM2.5 Air-Quality Risk Classification  
**Model:** Support Vector Machine (RBF Kernel)  
**Purpose:** Baseline model development and validation — Pabodha's model

This notebook follows the existing project methodology: chronological splitting, train-only preprocessing fit, validation evaluation, and an untouched test set. Hyperparameter tuning and imbalance optimization are intentionally left for later notebooks.

“A baseline model is the initial model used as a reference. Later, I compare the optimized model against this baseline to check whether optimization actually improves performance.”

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

project_root = Path.cwd().parent

# Handle cases where VS Code starts the notebook from a different working directory
if not (project_root / "data" / "processed").exists():
    current = Path.cwd().resolve()

    for parent in [current] + list(current.parents):
        if (parent / "data" / "processed").exists() and (parent / "src").exists():
            project_root = parent
            break
# Make project source code importable
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preprocessing.preprocessing_pipeline import (
    prepare_base_dataframe,
    prepare_features_and_target,
    build_preprocessor
)
      
print("Imports loaded successfully.")
print("Project root:", project_root)

## 1. Load Chronological Split Datasets
Uses the same files created by `07_make_dataset.ipynb`: training up to 2012, validation 2013–2014, and test 2015 onward.

Why chronological split?

Because this is a future prediction problem. We train on past years and evaluate on later years, which better represents real-world prediction and helps avoid future information leaking into training.

why not random split?
Random splitting could mix future observations with past observations and make the evaluation unrealistic. Chronological splitting better represents real deployment.



In [ ]:
processed_dir = project_root / "data" / "processed"
train_path = processed_dir / "train_data.csv"
validation_path = processed_dir / "validation_data.csv"
test_path = processed_dir / "test_data.csv"

for path in [train_path, validation_path, test_path]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}\nRun 07_make_dataset.ipynb first.")

identifier_types = {"site_id":"string","state_code":"string","county_code":"string","site_num":"string"}
train_df = pd.read_csv(train_path, dtype=identifier_types, low_memory=False)
validation_df = pd.read_csv(validation_path, dtype=identifier_types, low_memory=False)
test_df = pd.read_csv(test_path, dtype=identifier_types, low_memory=False)
print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

## 2. Confirm the Chronological Split

In [ ]:
for name, dataset in [("Training",train_df),("Validation",validation_df),("Test",test_df)]:
    print(f"{name}: {dataset['year'].min()} - {dataset['year'].max()} ({len(dataset)} rows)")

## 3. Apply Base Cleaning to Each Split

In [ ]:
train_prepared = prepare_base_dataframe(train_df)
validation_prepared = prepare_base_dataframe(validation_df)
test_prepared = prepare_base_dataframe(test_df)
print("Base preprocessing completed.")

## 4. Separate Features and Target

In [ ]:
X_train, y_train = prepare_features_and_target(train_prepared)
X_val, y_val = prepare_features_and_target(validation_prepared)
X_test, y_test = prepare_features_and_target(test_prepared)

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:", X_val.shape, "| y_val:", y_val.shape)
print("X_test:", X_test.shape, "| y_test:", y_test.shape)
print("\nTraining target distribution:")
print(y_train.value_counts())
print("\nTraining target percentages:")
print((y_train.value_counts(normalize=True)*100).round(2))

## 5. Build and Fit the Preprocessor
The transformer is fitted only on `X_train`. Scaling is important for the distance-based RBF kernel.

In [ ]:
preprocessor = build_preprocessor(X_train)
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)
print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

## 6. Train Baseline SVM (RBF)
Uses baseline `C=1.0` and `gamma="scale"`. No SMOTE or class weighting is used here.

In [ ]:
svm_rbf_baseline = SVC(kernel="rbf", C=1.0, gamma="scale")
svm_rbf_baseline.fit(X_train_processed, y_train)
print("Baseline SVM (RBF) training completed.")

## 7. Evaluate on the Original Validation Set

In [ ]:
y_val_pred = svm_rbf_baseline.predict(X_val_processed)
validation_accuracy = accuracy_score(y_val, y_val_pred)
validation_macro_f1 = f1_score(y_val, y_val_pred, average="macro", zero_division=0)

print("SVM (RBF) BASELINE - VALIDATION RESULTS")
print("="*50)
print(f"Accuracy : {validation_accuracy:.6f}")
print(f"Macro F1 : {validation_macro_f1:.6f}")

In [ ]:
classification_report_df = pd.DataFrame(
    classification_report(y_val, y_val_pred, output_dict=True, zero_division=0)
).T
classification_report_df

## 8. Validation Confusion Matrix

In [ ]:
pictures_dir = project_root / "reports" / "pictures"
pictures_dir.mkdir(parents=True, exist_ok=True)
labels = sorted(pd.Series(y_val).dropna().unique())
fig, ax = plt.subplots(figsize=(9,7))
ConfusionMatrixDisplay.from_predictions(y_val, y_val_pred, labels=labels, xticks_rotation=45, ax=ax)
ax.set_title("SVM (RBF) Baseline - Validation Confusion Matrix")
plt.tight_layout()
confusion_matrix_path = pictures_dir / "svm_rbf_baseline_confusion_matrix.png"
plt.savefig(confusion_matrix_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", confusion_matrix_path)

## 9. Validation Metrics Plot

In [ ]:
metrics_df = pd.DataFrame({"Metric":["Accuracy","Macro F1"],"Score":[validation_accuracy,validation_macro_f1]})
ax = metrics_df.plot(x="Metric",y="Score",kind="bar",legend=False,figsize=(7,5),rot=0)
ax.set_title("SVM (RBF) Baseline - Validation Metrics")
ax.set_ylabel("Score")
ax.set_ylim(0,1)
plt.tight_layout()
metrics_plot_path = pictures_dir / "svm_rbf_baseline_metrics.png"
plt.savefig(metrics_plot_path,dpi=300,bbox_inches="tight")
plt.show()
print("Saved:", metrics_plot_path)

## 10. Save Baseline Results

In [ ]:
results_dir = project_root / "reports" / "model_results"
results_dir.mkdir(parents=True, exist_ok=True)
summary_df = pd.DataFrame([{
    "model":"SVM (RBF) Baseline","kernel":"rbf","C":1.0,"gamma":"scale",
    "validation_accuracy":validation_accuracy,"validation_macro_f1":validation_macro_f1
}])
results_path = results_dir / "svm_rbf_baseline_results.csv"
report_path = results_dir / "svm_rbf_baseline_classification_report.csv"
summary_df.to_csv(results_path,index=False)
classification_report_df.to_csv(report_path)
print("Saved:",results_path)
print("Saved:",report_path)

## 11. Output Check

In [ ]:
for path in [results_path,report_path,confusion_matrix_path,metrics_plot_path]:
    print(("OK" if path.exists() else "MISSING"), "-", path)
print("\nIMPORTANT: Preprocessing was fitted only on training data.")
print("IMPORTANT: No SMOTE or class weighting was used in this baseline.")
print("IMPORTANT: Test labels were not used for model selection.")